# VisionEye Object Mapping

**YOLO Multi-Task Computer Vision Assignment — Task 3**

## Objective

Map tracked people to a fixed observation point in video using
Ultralytics VisionEye, pretrained YOLO26n and ByteTrack.

The full video processing experiment completed successfully.


## Execution and reproducibility

Use Google Colab with Google Drive mounted for the saved inputs and outputs. Install dependencies using the setup cells. The notebook contains experimental steps that may download media, run inference or train a model; rerun only the stages you intend and ensure required input paths exist. Results were produced in the documented Colab environment; a clean end-to-end rerun on a fresh runtime has not been certified.

Notebook outputs are cleared for safe version control. Results and limitations are documented below and in `README.md`.


In [ ]:
%pip install "ultralytics==8.4.208" "opencv-python-headless" "lap"

In [ ]:

from pathlib import Path
import torch
import cv2
import ultralytics
from ultralytics import YOLO, solutions
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/YOLO_Multi_Task")

VIDEO = BASE / "object_counting/videos/street_walk.mp4"
OUTPUT = BASE / "visioneye"
OUTPUT.mkdir(parents=True, exist_ok=True)

assert VIDEO.is_file(), "Source video not found"
assert hasattr(solutions, "VisionEye")

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened()

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

VISION_POINT = (width // 2, height - 80)

print("===== VISIONEYE SETUP =====")
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("Video:", VIDEO.name)
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Frames:", frames)
print("Vision point:", VISION_POINT)
print("VISIONEYE SETUP: PASS")


In [ ]:

from ultralytics import solutions
from pathlib import Path
import cv2
import json
import torch

RUN_DIR = OUTPUT / "street_walk_visioneye"
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = RUN_DIR / "visioneye_output.mp4"
REPORT = RUN_DIR / "visioneye_results.json"

assert VIDEO.is_file(), "Source video not found"
assert not OUTPUT_VIDEO.exists(), "Output video already exists"
assert not REPORT.exists(), "Report already exists"

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), "Cannot open video"

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

assert writer.isOpened(), "Cannot create output video"

visioneye = solutions.VisionEye(
    model="yolo26n.pt",
    vision_point=VISION_POINT,
    tracker="bytetrack.yaml",
    classes=[0],
    conf=0.25,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    show=False,
    verbose=False
)

frames_processed = 0
last_result = None

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        last_result = visioneye(frame)
        writer.write(last_result.plot_im)
        frames_processed += 1

finally:
    cap.release()
    writer.release()

assert frames_processed > 0
assert last_result is not None
assert OUTPUT_VIDEO.is_file()
assert OUTPUT_VIDEO.stat().st_size > 0

summary = {
    "task": "VisionEye Object Mapping",
    "model": "yolo26n.pt",
    "tracker": "bytetrack.yaml",
    "target_class": "person",
    "vision_point": list(VISION_POINT),
    "processed_frames": frames_processed,
    "fps": fps,
    "resolution": [width, height],
    "source_video": str(VIDEO),
    "output_video": str(OUTPUT_VIDEO)
}

REPORT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n========== VISIONEYE RESULTS ==========")
print("Processed frames:", frames_processed)
print("Vision point:", VISION_POINT)
print("Output video:", OUTPUT_VIDEO)
print("Report:", REPORT)
print("\nVISIONEYE INFERENCE: PASS")


In [ ]:
from pathlib import Path
from IPython.display import Video, display
import subprocess

RUN_DIR = OUTPUT / "street_walk_visioneye"

SOURCE = RUN_DIR / "visioneye_output.mp4"
PREVIEW = RUN_DIR / "visioneye_preview_h264.mp4"

assert SOURCE.is_file(), "VisionEye output video not found"

if not PREVIEW.is_file():
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(SOURCE),
            "-vf", "scale=960:-2",
            "-c:v", "libx264",
            "-preset", "fast",
            "-crf", "23",
            "-pix_fmt", "yuv420p",
            "-an",
            str(PREVIEW)
        ],
        check=True,
        capture_output=True,
        text=True
    )

assert PREVIEW.is_file() and PREVIEW.stat().st_size > 0

print("VisionEye Preview:", PREVIEW)
print("Size MB:", round(PREVIEW.stat().st_size / 1024**2, 2))

display(Video(str(PREVIEW), embed=True))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/visioneye"
)

RUN = BASE / "street_walk_visioneye"

required = {
    "source_video": Path(
        "/content/drive/MyDrive/YOLO_Multi_Task/"
        "object_counting/videos/street_walk.mp4"
    ),
    "output_video": RUN / "visioneye_output.mp4",
    "preview_video": RUN / "visioneye_preview_h264.mp4",
    "results": RUN / "visioneye_results.json",
}

for name, path in required.items():
    assert path.is_file(), f"Missing: {name}"
    print("OK:", name)

results = json.loads(required["results"].read_text(encoding="utf-8"))

assert results["processed_frames"] == 712

checkpoint = {
    "task": "03 - VisionEye Mapping",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "technical_and_visual_tests_passed",
    "model": "yolo26n.pt",
    "tracker": "bytetrack.yaml",
    "vision_point": results["vision_point"],
    "processed_frames": results["processed_frames"],
    "artifacts": {
        key: str(path) for key, path in required.items()
    },
    "remaining": [
        "Finalize notebook documentation",
        "Review and commit code to GitHub"
    ],
    "next_task": "04 - Neural Ocean Detection"
}

CHECKPOINT = BASE / "TASK3_PROGRESS_CHECKPOINT.json"

CHECKPOINT.write_text(
    json.dumps(checkpoint, indent=2),
    encoding="utf-8"
)

print("\nTASK 3 CHECKPOINT: SAVED")
print(CHECKPOINT)

## Reproducibility setup
Run after dependency installation. Fixed seeds reduce variability but do not guarantee identical results across devices. Save dependency versions, hardware, source video identity, dataset versions, and final configuration with each future experiment.

## 1. Video and observation point

Source: `street_walk.mp4`

- Resolution: 1920x1080
- Frames: 712
- FPS: approximately 29.97
- Observation point: `(960, 1000)`
- Target class: person (COCO class ID 0)


## 2. VisionEye configuration

The experiment used:

- `solutions.VisionEye`
- `yolo26n.pt`
- `bytetrack.yaml`
- Confidence threshold: 0.25
- Ultralytics: 8.4.208
- `show=False` for Google Colab

No custom training was required.


## 3. Video processing and export

All 712 frames were processed successfully.
The resulting video contains object bounding boxes,
tracking identifiers and rays connecting detected
objects to the fixed observation point.

A full-resolution output video and an H.264 preview
were saved to Google Drive, together with a JSON report.


## 4. Interpretation and limitations

VisionEye provides image-plane visual mapping.
The rays do not represent calibrated distances,
real-world coordinates or 3D reconstruction.

Tracking instability and partial occlusions can
affect the displayed object associations.


## 5. Results and conclusions

**Technical and visual inference: PASS**

- Processed frames: 712
- Vision point: `(960, 1000)`
- Model: YOLO26n
- Tracker: ByteTrack
- Annotated video: saved
- JSON report: saved

Visual inspection confirmed bounding boxes, IDs
and rays converging on the fixed observation point.

### References

- [Ultralytics VisionEye](https://docs.ultralytics.com/guides/vision-eye/)
- [Ultralytics Models](https://docs.ultralytics.com/models/)
